In [ ]:
# Test get_valid_windows

windows = get_valid_windows(
    counts_gz, 
    chromosomes=[('chr8', 23_237_000, 23_239_000), ('chr8', 23_300_000, 23_302_000)], 
    window_overlap_bp=0, 
    window_size=1024, 
    maxgap=1000,
    max_windows=10
)

print(windows)

In [ ]:
# Train a model on a small dataset

# Initialize W&B run
wandb.init(project="footprint-to-procap", name="run3")

# Create the WandbLogger
wandb_logger = WandbLogger(
    project="footprint-to-procap",
    log_model=False,      # we’ll use our own checkpoint callback
)

# Create a checkpoint callback to save the best val_loss
checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="footprint-{epoch:02d}-{val_loss:.4f}",
    auto_insert_metric_name=False,
)

counts_gz = '../data/mesc_microC_3hrDMSO_chr8.counts.tsv.gz'
procap_bw = '../data/GSM2170014_Pro_mESC.ucsc_mm10.bw'

# Create training and validation windows. Exclude test region
regions = get_valid_windows(counts_gz, chromosomes=[('chr8', 0, 23_000_000), ('chr8', 24_000_000, 1e9)], 
    window_size=1024, maxgap=500)
print(f"Number of windows: {len(regions)}")
np.random.shuffle(regions)
train_regions, val_regions = regions[:int(len(regions)*0.8)], regions[int(len(regions)*0.8):]
print(f"Number of training windows: {len(train_regions)}")
print(f"Number of validation windows: {len(val_regions)}")

data_accessor_fn = partial(get_footprint_and_procap, counts_gz, procap_bw, avg_by_len)

dm = FootprintDataModule(train_regions=train_regions, val_regions=val_regions, test_regions=[], accessor_fn=data_accessor_fn, batch_size=64, num_workers=4)
model = UNetLightning(in_channels=1, base_filters=32)
trainer = Trainer(accelerator="mps", max_epochs=2, gradient_clip_val=1.0, logger=wandb_logger, callbacks=[checkpoint_cb])
trainer.fit(model, dm)

wandb.save(checkpoint_cb.best_model_path)


In [ ]:
# Predict test regions

model = UNetLightning.load_from_checkpoint("/Users/martin/projects/footprint-tools/code/wandb/run-20250513_130038-kn0kkg6u/files/footprint-to-procap/kn0kkg6u/checkpoints/footprint-01-0.0457.ckpt")
trainer = Trainer(accelerator="mps")

# Create a set of test regions
test_regions = [
    ("chr8", 23_237_000, 23_238_023), # Gins4
    ("chr8", 23_208_000, 23_209_023), # Gpat4
]

data_accessor_fn = partial(get_footprint_and_procap, counts_gz, procap_bw, avg_by_len)

# turn off shuffling and set batch_size=1
dm_test = FootprintDataModule(train_regions=[], val_regions=[], test_regions=test_regions, 
                              accessor_fn=data_accessor_fn,
                              batch_size=1, num_workers=1)

dm_test.setup()
test_loader = dm_test.test_dataloader()

# run predictions
preds = trainer.predict(model, test_loader)

trues, mus, xs = [], [], []
for batch, pred in zip(test_loader, preds):
    x, y_true = batch            # y_true: (1, 1024)
    mu  = pred["mu"]             # mu: (1, 1024)
    xs.append(x.squeeze().cpu().numpy())
    trues.append(y_true.squeeze().cpu().numpy())
    mus.append(mu.squeeze().cpu().numpy())


In [ ]:
# View predictions

i = 0

_, start_bp, _ = test_regions[i]
x = xs[i]  # (128, 1024)
mu = mus[i]  # (1024,)
true = trues[i]  # (1024,)

print("x shape:", x.squeeze().shape)
print("mu shape:", mu.shape)
print("true shape:", true.shape)

# define your fragment‐length index and genomic‐position columns
fragment_len_min = 25
fragment_len_max = 150
frag_lens = np.arange(fragment_len_min, fragment_len_max + 1)       
positions = np.arange(1024) + start_bp    

# build the DataFrame
df_fp = pd.DataFrame(x[1:127, :], index=frag_lens, columns=positions)

# similarly make a Series for the true procap
ser_procap = pd.Series(true, index=positions)
ser_pred_procap = pd.Series(mu, index=positions)

# Gins4 chr8:23226610-23237668 (-)
markers = {
    23_237_668: 'Gins4_TSS',
    #23_237_650: 'PRO-Cap peak',
}


# now call the plotter
plot_count_matrix(
    df_fp,
    named_positions=markers,
    tracks={'procap': ser_procap, 'pred_procap': ser_pred_procap},
    xtick_spacing=500
)

In [ ]:
# Experiment with different threshold values
thresholds = [10.0, 15.0, 20.0]
blob_results = {}

for threshold in thresholds:
    # Detect blobs with current threshold
    blobs = detect_blobs_matrix(
        footprint_matrix=df_fp,
        threshold=threshold,
        sigma=sigma,
        min_size=min_size
    )
    
    # Store results
    blob_results[threshold] = {
        'count': len(blobs),
        'blobs': blobs
    }
    
    # Print statistics
    print(f"Threshold: {threshold}")
    print(f"  Detected {len(blobs)} blobs")
    if not blobs.empty:
        print(f"  Mean size: {blobs['size'].mean():.1f} pixels")
        print(f"  Mean signal: {blobs['mean_signal'].mean():.2f}")
    print()
    
    # Plot the footprint matrix with detected blobs
    plot_count_matrix(
        df_fp,
        named_positions=markers,
        tracks={'procap': ser_procap},
        blobs=blobs,
        blob_marker='o',
        blob_color='white',
        blob_size=20,
        xtick_spacing=500,
        title=f"Threshold: {threshold}, Detected {len(blobs)} blobs"
    )

In [ ]:
# Experiment with different sigma values
sigmas = [0.5, 1.0, 2.0]
threshold = 15.0  # Use a fixed threshold

for sigma in sigmas:
    # Detect blobs with current sigma
    blobs = detect_blobs_matrix(
        footprint_matrix=df_fp,
        threshold=threshold,
        sigma=sigma,
        min_size=min_size
    )
    
    # Print statistics
    print(f"Sigma: {sigma}")
    print(f"  Detected {len(blobs)} blobs")
    if not blobs.empty:
        print(f"  Mean size: {blobs['size'].mean():.1f} pixels")
        print(f"  Mean signal: {blobs['mean_signal'].mean():.2f}")
    print()
    
    # Plot the footprint matrix with detected blobs
    plot_count_matrix(
        df_fp,
        named_positions=markers,
        tracks={'procap': ser_procap},
        blobs=blobs,
        blob_marker='o',
        blob_color='white',
        blob_size=20,
        xtick_spacing=500,
        title=f"Sigma: {sigma}, Detected {len(blobs)} blobs"
    )